Adds a day/night column to dataset using the start time variable (before or after 5pm local)

In [ ]:
import os
from glob import glob

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

# ============================================================
# PARAMETERS
# ============================================================
DATA_DIR = os.path.join('..', 'data')
PATTERN = '*_data_*.csv'
OVERWRITE_EXISTING_OUTPUTS = False
DAY_START_CUTOFF_HOUR = 17

dataset_paths = sorted(glob(os.path.join(DATA_DIR, PATTERN)))
if not OVERWRITE_EXISTING_OUTPUTS:
    dataset_paths = [p for p in dataset_paths if not p.endswith('_day_night.csv')]

if not dataset_paths:
    raise ValueError(f'No datasets found in {os.path.abspath(DATA_DIR)} matching {PATTERN}')

print(f'Data dir: {os.path.abspath(DATA_DIR)}')
print(f'Found {len(dataset_paths)} dataset files')
print(f'Day games are defined as start_hour < {DAY_START_CUTOFF_HOUR}')

In [ ]:
summary_rows = []

for input_path in dataset_paths:
    dataset_file = os.path.basename(input_path)
    base, ext = os.path.splitext(dataset_file)
    output_file = f'{base}_day_night{ext}'
    output_path = os.path.join(DATA_DIR, output_file)

    df = pd.read_csv(input_path)

    if 'start_hour' in df.columns:
        start_hour = pd.to_numeric(df['start_hour'], errors='coerce')
    elif 'game_start' in df.columns:
        game_start = pd.to_datetime(df['game_start'], errors='coerce')
        start_hour = game_start.dt.hour
    else:
        print(f'Skipping {dataset_file}: missing start_hour and game_start')
        continue

    df['start_hour'] = start_hour
    df['day_night'] = np.where(df['start_hour'] < DAY_START_CUTOFF_HOUR, 'day', 'night')
    df.loc[df['start_hour'].isna(), 'day_night'] = pd.NA

    df.to_csv(output_path, index=False)

    counts = df['day_night'].value_counts(dropna=False)
    summary_rows.append({
        'dataset_file': dataset_file,
        'output_file': output_file,
        'rows': len(df),
        'day_games': int(counts.get('day', 0)),
        'night_games': int(counts.get('night', 0)),
        'missing_label': int(df['day_night'].isna().sum()),
    })

summary = pd.DataFrame(summary_rows).sort_values('dataset_file').reset_index(drop=True)
print(f'Processed {len(summary)} datasets')
summary

In [ ]:
summary_path = os.path.join(DATA_DIR, 'day_night_processing_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'Saved summary: {os.path.abspath(summary_path)}')